# Notebook 54 — SPECTER Embeddings for Citation Impact Prediction

**Hypothesis**: TF-IDF is a bag-of-words model that ignores word order and semantics. SPECTER (`allenai-specter`) is a transformer model pre-trained on 146k scientific papers using citation signals as supervision — papers that cite each other should have similar embeddings. Replacing TF-IDF with SPECTER embeddings should capture richer semantic structure in abstracts and improve F1.

**Prior art**:
- All previous text experiments used TF-IDF (5k or 10k features) — plateaued at ~51% clean F1.
- True clean baseline (nb51/nb53 REF-CLEAN): **51.28% F1 / 0.6654 AUC**
- Topic Prominence is the single strongest feature (2.3× next feature). Venue metrics critical.
- Simple models (LogisticRegression) consistently beat complex ones on this dataset.

**Approach**:
- Encode abstracts (and titles, if available) with `allenai-specter` → 768-d dense vectors
- Combine with venue/author numeric features (same 10 features from COL_MAP)
- Try LogisticRegression and LightGBM as classifiers
- Ablation: numeric-only (no text) with LightGBM to isolate contribution of each modality
- All labels: global threshold from training set only (no leakage, mirrors REF-CLEAN)

**Configs**:

| Config | Text features | Numeric | Classifier |
|--------|--------------|---------|------------|
| REF-CLEAN | TF-IDF 5k | yes | LogisticRegression |
| A | SPECTER 768-d | yes | LogisticRegression |
| B | SPECTER 768-d | yes | LightGBM |
| C | none | yes | LightGBM |
| D | SPECTER 768-d | no | LogisticRegression |


In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import copy

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
TRAIN_YEARS  = list(range(2010, 2018))
TEST_YEARS   = [2018, 2019, 2020]
QUANTILE     = 0.75

NB51_CLEAN_BASELINE_F1  = 0.5128
NB51_CLEAN_BASELINE_AUC = 0.6654

CACHE_DIR = Path('../../data/cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded')

## 1. Load data

In [ ]:
data_path = Path('../../data/processed/all_unis_cleaned.pkl')

if not data_path.exists():
    raise FileNotFoundError(
        f"Merged data not found: {data_path}\n"
        "Run notebooks 04 → 05 → 06 first to generate the merged dataset."
    )

df = pd.read_pickle(data_path)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nInstitution counts:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")

# Detect title column
TITLE_COL = None
for candidate in ['Title', 'title', 'Document title', 'Paper Title', 'paper_title']:
    if candidate in df.columns:
        TITLE_COL = candidate
        break
print(f"\nTitle column: {TITLE_COL!r}")
print(f"Abstract column: 'Abstract' present = {'Abstract' in df.columns}")

In [ ]:
# Temporal splits — AUB-only, same as REF-CLEAN in nb51/nb53
df_aub       = df[df['institution'] == 'AUB'].copy()
df_aub_train = df_aub[df_aub['Year'].isin(TRAIN_YEARS)].copy()
df_test_aub  = df_aub[df_aub['Year'].isin(TEST_YEARS)].copy()

print(f"AUB train (2010-2017): {len(df_aub_train):,}")
print(f"AUB test  (2018-2020): {len(df_test_aub):,}")

# Global labels — train threshold only (no leakage)
thr = df_aub_train['Citations'].quantile(QUANTILE)
y_train = (df_aub_train['Citations'] >= thr).astype(int)
y_test  = (df_test_aub['Citations']  >= thr).astype(int)
print(f"\nGlobal threshold: {thr:.0f} citations")
print(f"Train positive rate: {y_train.mean():.1%}")
print(f"Test  positive rate: {y_test.mean():.1%}")

## 2. Feature helpers

In [ ]:
COL_MAP = {
    'snip':             'SNIP (publication year)',
    'snip_pct':         'SNIP percentile',
    'citescore':        'CiteScore (publication year)',
    'citescore_pct':    'CiteScore percentile',
    'sjr':              'SJR (publication year)',
    'sjr_pct':          'SJR percentile',
    'topic_prom':       'Topic Prominence Percentile',
    'num_authors':      'Authors',
    'num_institutions': 'Affiliations',
    'num_countries':    'Countries',
}

def extract_numeric_features(subset_df):
    """Extract venue/author numeric features (same 10 as prior notebooks)."""
    vf = pd.DataFrame(index=subset_df.index)
    for feat, col in COL_MAP.items():
        if col in subset_df.columns:
            if col in ('Authors', 'Affiliations', 'Countries'):
                vf[feat] = subset_df[col].fillna('').apply(
                    lambda x: len(str(x).split(';')) if x else 1
                )
            else:
                vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
    return vf


def get_numeric_features(df_tr, df_te):
    """Return scaled numeric feature arrays."""
    vf_tr = extract_numeric_features(df_tr)
    vf_te = extract_numeric_features(df_te)
    tr_median = vf_tr.median()
    vf_tr = vf_tr.fillna(tr_median)
    vf_te = vf_te.fillna(tr_median)
    scaler = StandardScaler()
    X_tr_num = pd.DataFrame(scaler.fit_transform(vf_tr), index=vf_tr.index, columns=vf_tr.columns)
    X_te_num = pd.DataFrame(scaler.transform(vf_te),     index=vf_te.index, columns=vf_te.columns)
    return X_tr_num, X_te_num


def build_tfidf_features(df_tr, df_te):
    """TF-IDF baseline features (5k, same config as prior notebooks)."""
    tfidf = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2),
        min_df=5, max_df=0.8, stop_words='english'
    )
    prep = lambda s: str(s).lower() if pd.notna(s) else ''
    tr_mat = tfidf.fit_transform(df_tr['Abstract'].apply(prep))
    te_mat = tfidf.transform(df_te['Abstract'].apply(prep))
    cols   = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]
    return (
        pd.DataFrame(tr_mat.toarray(), index=df_tr.index, columns=cols),
        pd.DataFrame(te_mat.toarray(), index=df_te.index, columns=cols)
    )


def evaluate(model, X_tr, y_tr, X_te, y_te, label='', scale=False):
    """Fit model, grid-search threshold, return metrics dict."""
    if scale:
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    thresholds = np.arange(0.10, 0.91, 0.01)
    f1s    = [f1_score(y_te, (proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    y_pred = (proba >= best_t).astype(int)
    return {
        'label':          label,
        'f1':             f1_score(y_te, y_pred, zero_division=0),
        'auc':            roc_auc_score(y_te, proba),
        'recall':         recall_score(y_te, y_pred, zero_division=0),
        'precision':      precision_score(y_te, y_pred, zero_division=0),
        'threshold':      best_t,
        'n_train':        len(y_tr),
        'n_test':         len(y_te),
        'pos_rate_train': float(y_tr.mean()),
        'pos_rate_test':  float(y_te.mean()),
    }


print('Feature helpers ready.')

## 3. SPECTER embeddings

`allenai-specter` was trained on 146k scientific papers using a triplet loss over citation graphs: a paper's embedding should be closer to papers it cites than to random papers. The model takes `title [SEP] abstract` as input and outputs a 768-dimensional vector.

Embeddings are cached to disk after the first run.

In [ ]:
from sentence_transformers import SentenceTransformer
import pickle

SPECTER_CACHE = CACHE_DIR / 'specter_embeddings_aub.pkl'


def make_specter_input(subset_df, title_col=None):
    """Build title [SEP] abstract strings for SPECTER input."""
    abstracts = subset_df['Abstract'].fillna('').astype(str).str.lower()
    if title_col and title_col in subset_df.columns:
        titles = subset_df[title_col].fillna('').astype(str)
        return (titles + ' [SEP] ' + abstracts).tolist()
    return abstracts.tolist()


if SPECTER_CACHE.exists():
    print('Loading cached SPECTER embeddings...')
    with open(SPECTER_CACHE, 'rb') as f:
        cache = pickle.load(f)
    emb_train = cache['train']
    emb_test  = cache['test']
    print(f'  Train: {emb_train.shape}  |  Test: {emb_test.shape}')
else:
    print('Downloading and running SPECTER (first run — will cache result)...')
    specter = SentenceTransformer('allenai-specter')

    train_texts = make_specter_input(df_aub_train, TITLE_COL)
    test_texts  = make_specter_input(df_test_aub,  TITLE_COL)

    print(f'  Encoding {len(train_texts)} train papers...')
    emb_train = specter.encode(
        train_texts, batch_size=32, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=False
    )
    print(f'  Encoding {len(test_texts)} test papers...')
    emb_test = specter.encode(
        test_texts,  batch_size=32, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=False
    )

    with open(SPECTER_CACHE, 'wb') as f:
        pickle.dump({'train': emb_train, 'test': emb_test}, f)
    print(f'  Cached to {SPECTER_CACHE}')
    print(f'  Train: {emb_train.shape}  |  Test: {emb_test.shape}')

# Wrap in DataFrames with original indices
specter_cols = [f'sp_{i}' for i in range(emb_train.shape[1])]
df_sp_train = pd.DataFrame(emb_train, index=df_aub_train.index, columns=specter_cols)
df_sp_test  = pd.DataFrame(emb_test,  index=df_test_aub.index,  columns=specter_cols)
print('SPECTER embeddings ready.')

## 4. REF-CLEAN — TF-IDF + numeric, LogisticRegression (reproduce baseline)

In [ ]:
results = []

print('=' * 60)
print('REF-CLEAN: TF-IDF 5k + numeric, LogisticRegression')
print('=' * 60)

tfidf_tr, tfidf_te = build_tfidf_features(df_aub_train, df_test_aub)
num_tr, num_te     = get_numeric_features(df_aub_train, df_test_aub)

X_ref_tr = pd.concat([tfidf_tr, num_tr.set_index(tfidf_tr.index)], axis=1)
X_ref_te = pd.concat([tfidf_te, num_te.set_index(tfidf_te.index)], axis=1)

lr_model = LogisticRegression(
    max_iter=1000, class_weight='balanced',
    C=1.0, solver='lbfgs', random_state=RANDOM_STATE
)
res = evaluate(copy.deepcopy(lr_model), X_ref_tr, y_train, X_ref_te, y_test,
               label='REF-CLEAN (TF-IDF + numeric, LR)')
results.append(res)

print(f"  F1:  {res['f1']:.4f}  |  AUC: {res['auc']:.4f}")
print(f"  Recall: {res['recall']:.4f}  |  Precision: {res['precision']:.4f}")
print(f"  Expected ≈ {NB51_CLEAN_BASELINE_F1:.4f} (nb51 clean)")

## 5. Config A — SPECTER + numeric, LogisticRegression

In [ ]:
print('=' * 60)
print('Config A: SPECTER + numeric, LogisticRegression')
print('=' * 60)

# SPECTER embeddings are already dense floats; scale them
sp_scaler = StandardScaler()
sp_tr_scaled = pd.DataFrame(
    sp_scaler.fit_transform(df_sp_train),
    index=df_sp_train.index, columns=specter_cols
)
sp_te_scaled = pd.DataFrame(
    sp_scaler.transform(df_sp_test),
    index=df_sp_test.index, columns=specter_cols
)

X_a_tr = pd.concat([sp_tr_scaled, num_tr.set_index(sp_tr_scaled.index)], axis=1)
X_a_te = pd.concat([sp_te_scaled, num_te.set_index(sp_te_scaled.index)], axis=1)

res_a = evaluate(copy.deepcopy(lr_model), X_a_tr, y_train, X_a_te, y_test,
                 label='Config A (SPECTER + numeric, LR)')
results.append(res_a)

delta_f1  = res_a['f1']  - results[0]['f1']
delta_auc = res_a['auc'] - results[0]['auc']
print(f"  F1:  {res_a['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC: {res_a['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Recall: {res_a['recall']:.4f}  |  Precision: {res_a['precision']:.4f}")

## 6. Config B — SPECTER + numeric, LightGBM

In [ ]:
print('=' * 60)
print('Config B: SPECTER + numeric, LightGBM')
print('=' * 60)

n_pos   = int(y_train.sum())
n_neg   = int((y_train == 0).sum())
scale_w = n_neg / n_pos  # class imbalance weight

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_w,
    random_state=RANDOM_STATE,
    verbose=-1,
)

# LightGBM doesn't need scaling but use same arrays as Config A for consistency
res_b = evaluate(copy.deepcopy(lgbm_model), X_a_tr.values, y_train, X_a_te.values, y_test,
                 label='Config B (SPECTER + numeric, LGBM)')
results.append(res_b)

delta_f1  = res_b['f1']  - results[0]['f1']
delta_auc = res_b['auc'] - results[0]['auc']
print(f"  F1:  {res_b['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC: {res_b['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Recall: {res_b['recall']:.4f}  |  Precision: {res_b['precision']:.4f}")

## 7. Config C — Numeric-only, LightGBM (ablation: no text)

In [ ]:
print('=' * 60)
print('Config C: Numeric-only (no text), LightGBM')
print('=' * 60)

res_c = evaluate(copy.deepcopy(lgbm_model), num_tr.values, y_train, num_te.values, y_test,
                 label='Config C (numeric-only, LGBM)')
results.append(res_c)

delta_f1  = res_c['f1']  - results[0]['f1']
delta_auc = res_c['auc'] - results[0]['auc']
print(f"  F1:  {res_c['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC: {res_c['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Recall: {res_c['recall']:.4f}  |  Precision: {res_c['precision']:.4f}")
print(f"  (n_features: {num_tr.shape[1]})")

## 8. Config D — SPECTER-only, LogisticRegression (ablation: no numeric)

In [ ]:
print('=' * 60)
print('Config D: SPECTER-only (no numeric), LogisticRegression')
print('=' * 60)

res_d = evaluate(copy.deepcopy(lr_model), sp_tr_scaled, y_train, sp_te_scaled, y_test,
                 label='Config D (SPECTER-only, LR)')
results.append(res_d)

delta_f1  = res_d['f1']  - results[0]['f1']
delta_auc = res_d['auc'] - results[0]['auc']
print(f"  F1:  {res_d['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC: {res_d['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Recall: {res_d['recall']:.4f}  |  Precision: {res_d['precision']:.4f}")

## 9. Config E — SPECTER + numeric, LightGBM with hyperparameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

print('=' * 60)
print('Config E: SPECTER + numeric, LightGBM (tuned)')
print('=' * 60)

param_dist = {
    'n_estimators':      [200, 300, 500, 700],
    'learning_rate':     [0.01, 0.03, 0.05, 0.1],
    'num_leaves':        [15, 31, 63, 127],
    'min_child_samples': [10, 20, 30, 50],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'reg_alpha':         [0.0, 0.1, 0.5, 1.0],
    'reg_lambda':        [0.0, 0.1, 0.5, 1.0],
}

base_lgbm = LGBMClassifier(
    scale_pos_weight=scale_w,
    random_state=RANDOM_STATE,
    verbose=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    base_lgbm, param_dist,
    n_iter=50, scoring='f1', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
search.fit(X_a_tr.values, y_train)

print(f"  Best CV F1: {search.best_score_:.4f}")
print(f"  Best params: {search.best_params_}")

best_lgbm = search.best_estimator_
res_e = evaluate(best_lgbm, X_a_tr.values, y_train, X_a_te.values, y_test,
                 label='Config E (SPECTER + numeric, LGBM tuned)')
results.append(res_e)

delta_f1  = res_e['f1']  - results[0]['f1']
delta_auc = res_e['auc'] - results[0]['auc']
print(f"  F1:  {res_e['f1']:.4f}  ({delta_f1:+.4f} vs REF-CLEAN)")
print(f"  AUC: {res_e['auc']:.4f}  ({delta_auc:+.4f} vs REF-CLEAN)")
print(f"  Recall: {res_e['recall']:.4f}  |  Precision: {res_e['precision']:.4f}")

## 10. Results summary

In [ ]:
import matplotlib.pyplot as plt

res_df = pd.DataFrame(results)
ref_f1  = res_df.loc[res_df['label'].str.startswith('REF'), 'f1'].values[0]
ref_auc = res_df.loc[res_df['label'].str.startswith('REF'), 'auc'].values[0]
res_df['delta_f1']  = res_df['f1']  - ref_f1
res_df['delta_auc'] = res_df['auc'] - ref_auc

print('\n' + '=' * 100)
print('RESULTS SUMMARY — Notebook 54: SPECTER Embeddings')
print('=' * 100)
cols = ['label', 'f1', 'delta_f1', 'auc', 'delta_auc', 'recall', 'precision', 'threshold',
        'pos_rate_train', 'pos_rate_test']
print(res_df[cols].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nnb51 clean baseline: F1={NB51_CLEAN_BASELINE_F1:.4f}  AUC={NB51_CLEAN_BASELINE_AUC:.4f}")

best = res_df.loc[res_df['f1'].idxmax()]
print(f"\nBest config: {best['label']}")
print(f"  F1:  {best['f1']:.4f}  ({best['delta_f1']:+.4f} vs REF-CLEAN)")
print(f"  AUC: {best['auc']:.4f}  ({best['delta_auc']:+.4f} vs REF-CLEAN)")
print(f"  Recall: {best['recall']:.4f}  |  Precision: {best['precision']:.4f}")

# Key interpretation
print('\n--- Key checks ---')
for _, row in res_df.iterrows():
    pos_ok = abs(row['pos_rate_test'] - 0.25) < 0.05  # test positive rate near true 25%
    verdict = 'OK' if pos_ok else 'CHECK pos_rate'
    sign = '+' if row['delta_f1'] >= 0 else ''
    outcome = 'IMPROVED' if row['delta_f1'] > 0.01 else ('DEGRADED' if row['delta_f1'] < -0.01 else 'FLAT')
    print(f"  {outcome:8s}  {verdict:15s}  {row['label']:50s}"
          f"  F1={row['f1']:.4f}({sign}{row['delta_f1']:.4f})"
          f"  AUC={row['auc']:.4f}({sign}{row['delta_auc']:.4f})"
          f"  pos_test={row['pos_rate_test']:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#4C8BE2' if d >= 0 else '#E24C4C' for d in res_df['delta_f1']]
labels_short = [l.split('(')[0].strip() for l in res_df['label']]

# F1
axes[0].barh(labels_short, res_df['f1'], color=colors)
axes[0].axvline(NB51_CLEAN_BASELINE_F1, color='red', linestyle='--', linewidth=1.5, label='nb51 baseline')
axes[0].axvline(0.75, color='green', linestyle=':', linewidth=1.5, label='Target 0.75')
axes[0].set_title('F1 Score')
axes[0].legend(fontsize=8)
for i, (v, d) in enumerate(zip(res_df['f1'], res_df['delta_f1'])):
    axes[0].text(v + 0.002, i, f'{v:.4f} ({d:+.4f})', va='center', fontsize=8)

# AUC
axes[1].barh(labels_short, res_df['auc'], color='#4C8BE2')
axes[1].axvline(NB51_CLEAN_BASELINE_AUC, color='red', linestyle='--', linewidth=1.5, label='nb51 baseline')
axes[1].set_title('ROC-AUC')
axes[1].legend(fontsize=8)
for i, v in enumerate(res_df['auc']):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

# Precision vs Recall
axes[2].scatter(res_df['recall'], res_df['precision'], s=80, color=colors, zorder=3)
for _, row in res_df.iterrows():
    lbl = row['label'].split('(')[0].strip()
    axes[2].annotate(lbl, (row['recall'], row['precision']),
                     textcoords='offset points', xytext=(5, 2), fontsize=7)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision vs Recall')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Notebook 54 — SPECTER Embeddings: Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../../docs/nb54_specter_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 11. Feature importance (best LGBM config)

In [ ]:
# Show which SPECTER dimensions and numeric features matter most
best_label = res_df.loc[res_df['f1'].idxmax(), 'label']
print(f"Feature importance for: {best_label}")

# Use Config B model (SPECTER + numeric, untuned LGBM) for interpretability
# Re-fit to get the model object back
lgbm_fi = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_w, random_state=RANDOM_STATE, verbose=-1
)
lgbm_fi.fit(X_a_tr.values, y_train)
feature_names = list(X_a_tr.columns)
importances   = lgbm_fi.feature_importances_

fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False)

# Aggregate SPECTER dimensions vs numeric features
fi_df['type'] = fi_df['feature'].apply(lambda x: 'SPECTER' if x.startswith('sp_') else 'Numeric')
print("\nAggregate importance by feature type:")
print(fi_df.groupby('type')['importance'].agg(['sum', 'mean', 'count']).to_string())

print("\nTop 20 individual features:")
print(fi_df.head(20).to_string(index=False))

# Top numeric features specifically
print("\nNumeric feature importances:")
print(fi_df[fi_df['type'] == 'Numeric'].to_string(index=False))

## 12. Conclusions

In [ ]:
print('=' * 70)
print('NOTEBOOK 54 — CONCLUSIONS')
print('=' * 70)

best = res_df.loc[res_df['f1'].idxmax()]
best_auc_row = res_df.loc[res_df['auc'].idxmax()]

print(f"\nREF-CLEAN (TF-IDF baseline):")
ref_row = res_df[res_df['label'].str.startswith('REF')].iloc[0]
print(f"  F1={ref_row['f1']:.4f}  AUC={ref_row['auc']:.4f}")

print(f"\nBest F1 config: {best['label']}")
print(f"  F1:  {best['f1']:.4f}  ({best['delta_f1']:+.4f} vs REF-CLEAN)")
print(f"  AUC: {best['auc']:.4f}  ({best['delta_auc']:+.4f} vs REF-CLEAN)")

print(f"\nBest AUC config: {best_auc_row['label']}")
print(f"  AUC: {best_auc_row['auc']:.4f}  ({best_auc_row['delta_auc']:+.4f} vs REF-CLEAN)")

print(f"\nnb51 clean reference: F1={NB51_CLEAN_BASELINE_F1:.4f}  AUC={NB51_CLEAN_BASELINE_AUC:.4f}")
print(f"Supervisor target:    F1=0.7500")
print(f"Gap to target:        {0.75 - best['f1']:+.4f}")

print('\n--- Config summary ---')
for _, row in res_df.iterrows():
    sign = '+' if row['delta_f1'] >= 0 else ''
    outcome = 'IMPROVED' if row['delta_f1'] > 0.01 else ('DEGRADED' if row['delta_f1'] < -0.01 else 'FLAT')
    print(f"  {outcome:8s}  {row['label']:52s}"
          f"  F1={row['f1']:.4f}({sign}{row['delta_f1']:.4f})"
          f"  AUC={row['auc']:.4f}({sign}{row['delta_auc']:.4f})")